In [1]:
import torch 
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader
from PIL import Image
from sklearn.datasets import load_iris
import numpy as np ; import pandas as pd
from sklearn.model_selection import train_test_split
from typing import  List,Tuple,Dict
from torchmetrics.classification import MulticlassPrecision, MulticlassRecall


#
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
generator = torch.Generator(device=device)
#torch.set_default_device(device=device)

print(device)


/home/linux/miniconda3/envs/P_3_11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda:0


In [40]:
class SimpleCNN(torch.nn.Module):
    def __init__(self):
       
        super(SimpleCNN,self).__init__()
        
        self.net = torch.nn.Sequential(
            torch.nn.Conv2d(3,16,kernel_size=3,padding=1),
            torch.nn.BatchNorm2d(16),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2,2),

            torch.nn.Conv2d(16,64,kernel_size=3,padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2,2),

            
       
            torch.nn.Flatten(start_dim=1),
            torch.nn.LazyLinear(256),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.4),
            torch.nn.Linear(256,256),
            torch.nn.BatchNorm1d(256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.4),
            torch.nn.Linear(256,10),
        )
    def forward(self,X):
        return self.net(X)
    



In [41]:
def train_model(model,train_loader,criterion,optimizer,device):
    model.train()

    running_lose = 0.0

    for images,labels in train_loader:

        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_lose+= loss.item()
    return running_lose/len(train_loader)
    

In [42]:

def evalutae_model(model,test_loader,device):
    model.eval()
    correct = 0 
    total = 0
    with torch.no_grad():
        
        for images,labels in test_loader:
            images ,labels  = images.to(device),labels.to(device)
            outputs = model(images)

            _,predicted = torch.max(outputs,1)
            total += labels.size(0)
            correct+= (predicted==labels).sum().item()
    return 100 * correct/total


In [43]:
def load_data(data_augmentaion:bool = False):
    generator = torch.Generator(device='cpu')
    transforms_original = transforms.Compose([
        transforms.Resize((32,32)),
        transforms.ToTensor(),
        transforms.Normalize((.5,.5,.5),(.5,.5,.5))

    ])

    transforms_augmented = transforms.Compose([
        transforms.Resize((32,32)),
        transforms.RandomHorizontalFlip(0.4),
        transforms.RandomRotation(15),
        #transforms.RandomCrop(32,padding=4),
        transforms.RandomVerticalFlip(0.4),
        transforms.ToTensor(),
        transforms.Normalize((.5,.5,.5),[.5,.5,.5])

    ])
    if data_augmentaion:
        original_dataset = torchvision.datasets.CIFAR10('./CIFAR',train=True,download=True,transform=transforms_original)
        augmented_dataset = torchvision.datasets.CIFAR10('./CIFAR',train=True,download=True,transform=transforms_augmented)
        train_dataset = torch.utils.data.ConcatDataset([original_dataset,augmented_dataset])
    else:
        train_dataset = torchvision.datasets.CIFAR10('./CIFAR',train=True,download=True,transform=transforms_original)
    
    test_dataset = torchvision.datasets.CIFAR10('./CIFAR',train=False,download=True,transform=transforms_original)

    train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True,num_workers=0,pin_memory=True,generator=generator)
    test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False,num_workers=0,pin_memory=True,generator=torch.Generator(device='cpu'))

    return train_loader,test_loader


In [44]:
epochs = 50 

train_loader,test_loader = load_data(data_augmentaion=False)

model=SimpleCNN().to(device=device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr=0.01)

print('No Augmentation')
loss_arr = []
for e in range(epochs):
    loss = train_model(model,train_loader,criterion,optimizer,device)
    loss_arr.append(loss)
    print(f'Epoch {e+1}  has a los of {loss}')
accuracy = evalutae_model(model,test_loader,device)

print(f'accuracy {accuracy}')


No Augmentation
Epoch 1  has a los of 1.3861061334609985
Epoch 2  has a los of 1.0516186986128082
Epoch 3  has a los of 0.9271762056271439
Epoch 4  has a los of 0.8405436286154915
Epoch 5  has a los of 0.7745442664074471
Epoch 6  has a los of 0.7191234199363558
Epoch 7  has a los of 0.6633355069876937
Epoch 8  has a los of 0.6207445306927347
Epoch 9  has a los of 0.5885652437082032
Epoch 10  has a los of 0.5530122911648068
Epoch 11  has a los of 0.5273646325101633
Epoch 12  has a los of 0.500171536889375
Epoch 13  has a los of 0.4731033967850763
Epoch 14  has a los of 0.4653126762422454
Epoch 15  has a los of 0.4483388573159952
Epoch 16  has a los of 0.4330436521593262
Epoch 17  has a los of 0.4259331537901288
Epoch 18  has a los of 0.4258334580856516
Epoch 19  has a los of 0.4049213799788519
Epoch 20  has a los of 0.3984337259474618
Epoch 21  has a los of 0.3975933218170005
Epoch 22  has a los of 0.38922118471787714
Epoch 23  has a los of 0.3801226888680854
Epoch 24  has a los of 0.38

In [49]:
epochs = 50 

train_loader,test_loader = load_data(data_augmentaion=True)

model=SimpleCNN().to(device=device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(),lr=0.01)

print('Augmentation is On')
loss_arr = []
for e in range(epochs):
    loss = train_model(model,train_loader,criterion,optimizer,device)
    loss_arr.append(loss)
    print(f'Epoch {e+1}  has a los of {loss}')
accuracy = evalutae_model(model,test_loader,device)

print(f'accuracy {accuracy}')


/home/linux/miniconda3/envs/P_3_11/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Augmentation is On
Epoch 1  has a los of 1.4015708458171925
Epoch 2  has a los of 1.147601283786393
Epoch 3  has a los of 1.048789585353622
Epoch 4  has a los of 0.985310666124865
Epoch 5  has a los of 0.9449994992691206
Epoch 6  has a los of 0.9160204989667589
Epoch 7  has a los of 0.888895202087273
Epoch 8  has a los of 0.8712323629657809
Epoch 9  has a los of 0.8560632681205954
Epoch 10  has a los of 0.8404397485504834
Epoch 11  has a los of 0.8257570703366744
Epoch 12  has a los of 0.8208522605766337
Epoch 13  has a los of 0.8185936487110014
Epoch 14  has a los of 0.8067676030475973
Epoch 15  has a los of 0.8055550693817346
Epoch 16  has a los of 0.8004561762587047
Epoch 17  has a los of 0.7921914876803937
Epoch 18  has a los of 0.7879793793058365
Epoch 19  has a los of 0.7900282066560905
Epoch 20  has a los of 0.7868269242808671
Epoch 21  has a los of 0.7864670994300074
Epoch 22  has a los of 0.774953362961572
Epoch 23  has a los of 0.7768220479718707
Epoch 24  has a los of 0.7770

In [46]:
accuracy = evalutae_model(model,train_loader,device)

In [47]:
accuracy

87.073